# Phase 2.5 — PPO-V2 Recovery & Stabilization Training 🔥

This notebook trains PPO-V2 models with:
- **Enhanced reward shaping** (5 components: spread reduction, frontier blocking, coverage, overlap penalty, containment stability)
- **Entropy regularization sweep** (ent_coef: 0.005, 0.01, 0.02, 0.05)
- **Longer training budgets** (300k–1M steps per team size)

### Instructions:
1. Upload `wildfire-rl.zip` (without `models/` to keep it small)
2. Select **T4 GPU** runtime
3. Run all cells sequentially
4. Download trained models when complete

### 1. Extract Workspace Code

In [ ]:
import os
from pathlib import Path

zip_name = "wildfire-rl.zip"

if Path(zip_name).exists():
    !unzip -q {zip_name} -d wildfire-rl
    %cd wildfire-rl
    print(f"Successfully entered directory: {os.getcwd()}")
else:
    print(f"ERROR: Could not find '{zip_name}'. Please upload it first.")

### 2. Install Dependencies

In [ ]:
!grep -v "torch" requirements.txt | grep -v "numpy" > req_clean.txt
!pip install -q -r req_clean.txt
!pip install -q -e .

### 3. Verify GPU

*If you get a NumPy error, restart session (Runtime → Restart session), skip cell 2, resume from here.*

In [ ]:
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU. Go to Runtime → Change runtime type → T4 GPU.")

### 4. Entropy Coefficient Sweep (Saudi, 5 agents)

This cell trains 12 models: 4 ent_coef values × 3 seeds, each at 100k steps.

**Purpose**: Find the best `ent_coef` that maximizes action diversity without sacrificing performance.

In [ ]:
import subprocess, time
from pathlib import Path

ENT_COEFS = [0.005, 0.01, 0.02, 0.05]
SWEEP_TIMESTEPS = 100000
MAX_PARALLEL = 4

jobs = []
for ent_coef in ENT_COEFS:
    for seed in [0, 1, 2]:
        jobs.append(("saudi", 5, seed, ent_coef))

processes = []
print(f"Launching {len(jobs)} entropy sweep runs ({MAX_PARALLEL} parallel)...")

for region, agents, seed, ent_coef in jobs:
    model_file = Path("models") / f"ppo_v2_marl_{region}_{agents}agents_ent{ent_coef}_seed_{seed}.zip"
    if model_file.exists():
        print(f"Skip: {model_file.name} already exists.")
        continue

    while len(processes) >= MAX_PARALLEL:
        for p in list(processes):
            if p.poll() is not None:
                processes.remove(p)
        time.sleep(1)

    cmd = [
        "python", "scripts/train_single_marl_v2.py",
        "--region", region,
        "--agents", str(agents),
        "--seed", str(seed),
        "--timesteps", str(SWEEP_TIMESTEPS),
        "--ent-coef", str(ent_coef),
    ]
    print(f"Launching: ent_coef={ent_coef}, seed={seed}")
    p = subprocess.Popen(cmd)
    processes.append(p)

for p in processes:
    p.wait()

print("\n✅ Entropy sweep complete!")

### 5. Quick Sweep Analysis — Pick Best ent_coef

Evaluate sweep models to find the ent_coef with highest entropy + lowest fire intensity.

In [ ]:
import numpy as np
from wildfire_rl.config import load_config
from wildfire_rl.envs.multi_agent_v2 import MultiAgentWildfireEnvV2
from wildfire_rl.eval.loading import load_ppo_model
from wildfire_rl.paths import models_dir, region_tensor_path

cfg = load_config("configs/ppo/marl_v2.yaml")
tensor = np.load(region_tensor_path("saudi_eastern_province", 32))
env_cfg = cfg.env
env_cfg.reward_mode = "normalized"
env_cfg.reward_v2.enabled = True

ENT_COEFS = [0.005, 0.01, 0.02, 0.05]

print(f"{'ent_coef':>10} | {'Entropy':>8} | {'Repeated':>8} | {'Fire':>10}")
print("-" * 50)

best_ent = 0.02
best_score = -1e9

for ent_coef in ENT_COEFS:
    ents, reps, fires = [], [], []
    for seed in [0, 1, 2]:
        path = models_dir() / f"ppo_v2_marl_saudi_5agents_ent{ent_coef}_seed_{seed}.zip"
        if not path.exists():
            continue
        factory = lambda: MultiAgentWildfireEnvV2(state_tensor=tensor, config=env_cfg, num_agents=5)
        env = factory()
        model = load_ppo_model(path, env)

        obs, _ = env.reset(seed=seed)
        done = False
        actions = []
        while not done:
            action, _ = model.predict(obs, deterministic=True)
            actions.append(np.atleast_1d(action).copy())
            obs, _, terminated, truncated, _ = env.step(action)
            done = bool(terminated or truncated)

        acts = np.array(actions)
        for i in range(5):
            col = acts[:, i]
            _, counts = np.unique(col, return_counts=True)
            probs = counts / len(col)
            ents.append(-np.sum(probs * np.log2(probs + 1e-12)))
            reps.append(np.sum(col[:-1] == col[1:]) / max(len(col)-1, 1))
        fires.append(float(env.state[0].sum()))

    if ents:
        mean_ent = np.mean(ents)
        mean_rep = np.mean(reps)
        mean_fire = np.mean(fires)
        # Score: maximize entropy, minimize fire, penalize repetition
        score = mean_ent - mean_rep - mean_fire / 100
        print(f"{ent_coef:>10} | {mean_ent:>8.3f} | {mean_rep:>8.3f} | {mean_fire:>10.2f}")
        if score > best_score:
            best_score = score
            best_ent = ent_coef

print(f"\n🏆 Best ent_coef: {best_ent}")

### 6. Full V2 Training with Best ent_coef

Trains all V2 models:
- 2 regions × 3 team sizes × 3 seeds = 18 runs
- Budgets: 3 agents→300k, 5 agents→500k, 10 agents→1M

**⚠️ This takes ~1-2 hours on T4 GPU.**

In [ ]:
import subprocess, time
from pathlib import Path

# Use the best ent_coef from the sweep (or override manually)
BEST_ENT_COEF = best_ent  # From cell above
# BEST_ENT_COEF = 0.02  # Uncomment to override manually

TIMESTEPS_MAP = {3: 300000, 5: 500000, 10: 1000000}
MAX_PARALLEL = 4

jobs = []
for region in ["saudi", "california"]:
    for n_agents, timesteps in TIMESTEPS_MAP.items():
        for seed in [0, 1, 2]:
            jobs.append((region, n_agents, seed, timesteps))

processes = []
print(f"Launching {len(jobs)} V2 training runs ({MAX_PARALLEL} parallel)...")
print(f"Using ent_coef={BEST_ENT_COEF}")

for region, agents, seed, timesteps in jobs:
    model_file = Path("models") / f"ppo_v2_marl_{region}_{agents}agents_ent{BEST_ENT_COEF}_seed_{seed}.zip"
    if model_file.exists():
        print(f"Skip: {model_file.name}")
        continue

    while len(processes) >= MAX_PARALLEL:
        for p in list(processes):
            if p.poll() is not None:
                processes.remove(p)
        time.sleep(1)

    cmd = [
        "python", "scripts/train_single_marl_v2.py",
        "--region", region,
        "--agents", str(agents),
        "--seed", str(seed),
        "--timesteps", str(timesteps),
        "--ent-coef", str(BEST_ENT_COEF),
    ]
    print(f"Launching: {region}, {agents} agents, {timesteps} steps, seed={seed}")
    p = subprocess.Popen(cmd)
    processes.append(p)

for p in processes:
    p.wait()

print("\n✅ All V2 training complete!")

### 7. Run V2 Evaluation

In [ ]:
!python scripts/run_marl_v2_evaluation.py
print("\n✅ V2 evaluation complete. Check results/v2/ and figures/v2/ for outputs.")

### 8. Download All Trained Models & Results

In [ ]:
# Package V2 models + results + figures
!zip -j v2_models.zip models/ppo_v2_marl_*
!zip -r v2_results.zip results/v2/ figures/v2/

print("\n--- READY FOR DOWNLOAD ---")
print("1. Click file explorer in Colab (left panel).")
print("2. Download 'v2_models.zip' and 'v2_results.zip'.")
print("3. Extract v2_models.zip into your local 'models/' folder.")
print("4. Extract v2_results.zip into your repo root.")